In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/ecommerce.db")

conn = sqlite3.connect(db_path)

In [2]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
"""

pd.read_sql(query, conn)

,name
0,category_translation
1,customers
2,dim_customer
3,dim_product
4,dim_seller
5,fact_sales
6,fact_sales_final
7,fact_sales_final_v2
8,fact_sales_v2
9,fact_sales_v3


In [3]:
from pathlib import Path

export_path = Path("../data/tableau")

export_path.mkdir(exist_ok=True)

export_path

PosixPath('../data/tableau')

In [4]:
fact = pd.read_sql(
    "SELECT * FROM fact_sales_final_v2",
    conn
)

fact.to_csv(
    export_path / "fact_sales.csv",
    index=False
)

print(fact.shape)

(112834, 30)


In [5]:
customer = pd.read_sql(
    "SELECT * FROM dim_customer",
    conn
)

customer.to_csv(
    export_path / "dim_customer.csv",
    index=False
)

print(customer.shape)

(95539, 3)


In [6]:
product = pd.read_sql(
    "SELECT * FROM dim_product",
    conn
)

product.to_csv(
    export_path / "dim_product.csv",
    index=False
)

print(product.shape)

(32951, 6)


In [7]:
seller = pd.read_sql(
    "SELECT * FROM dim_seller",
    conn
)

seller.to_csv(
    export_path / "dim_seller.csv",
    index=False
)

print(seller.shape)

(3095, 3)


In [8]:
import os

os.listdir(export_path)

['dim_product.csv', 'dim_customer.csv', 'fact_sales.csv', 'dim_seller.csv']

In [9]:
import pandas as pd

fact = pd.read_csv("../data/tableau/fact_sales.csv")

print(fact["allocated_payment"].head(10))
print(fact["allocated_payment"].isna().sum())
print(fact["allocated_payment"].dtype)

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: allocated_payment, dtype: float64
112834
float64


In [10]:
fact[["allocated_payment"]].describe(include="all")

,allocated_payment
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [14]:
pd.read_sql("""
SELECT
COUNT(*) AS total_rows,
COUNT(allocated_payment) AS allocated_not_null,
COUNT(total_payment) AS payment_not_null
FROM fact_sales_final_v2;
""", conn)

,total_rows,allocated_not_null,payment_not_null
0,112834,0,112831


In [15]:
pd.read_sql("""
SELECT
price,
total_payment,
allocated_payment
FROM fact_sales_final_v2
LIMIT 10;
""", conn)

,price,total_payment,allocated_payment
0,58.90,72.19,None
1,239.90,259.83,None
2,199.00,216.87,None
3,12.99,25.78,None
4,199.90,218.04,None
5,21.90,34.59,None
6,19.90,31.75,None
7,810.00,880.75,None
8,145.95,157.60,None
9,53.99,65.39,None


In [16]:
pd.read_sql("""
SELECT
COUNT(*) AS rows,
COUNT(total_merchandise) AS non_null_merchandise,
MIN(total_merchandise) AS min_value,
MAX(total_merchandise) AS max_value
FROM order_totals;
""", conn)

,rows,non_null_merchandise,min_value,max_value
0,98666,98666,0.85,13440.0


In [17]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales_final_v3;

CREATE TABLE fact_sales_final_v3 AS

SELECT

    fs.*,

    ROUND(
        (fs.price * 1.0 / ot.total_merchandise) * fs.total_payment,
        2
    ) AS allocated_payment

FROM fact_sales_final fs
JOIN order_totals ot
ON fs.order_id = ot.order_id;

""")

conn.commit()

print("✅ fact_sales_final_v3 created")

✅ fact_sales_final_v3 created


In [18]:
pd.read_sql("""
SELECT
price,
total_payment,
allocated_payment
FROM fact_sales_final_v3
LIMIT 10;
""", conn)

,price,total_payment,allocated_payment
0,58.90,72.19,None
1,239.90,259.83,None
2,199.00,216.87,None
3,12.99,25.78,None
4,199.90,218.04,None
5,21.90,34.59,None
6,19.90,31.75,None
7,810.00,880.75,None
8,145.95,157.60,None
9,53.99,65.39,None


In [19]:
pd.read_sql("""
SELECT
price,
order_id
FROM fact_sales_final
LIMIT 10;
""", conn)

,price,order_id
0,58.90,00010242fe8c5a6d1ba2dd792cb16214
1,239.90,00018f77f2f0320c557190d7a144bdd3
2,199.00,000229ec398224ef6ca0657da4fc703e
3,12.99,00024acbcdf0a6daa1e931b038114c75
4,199.90,00042b26cf59d7ce69dfabb4e55b4fd9
5,21.90,00048cc3ae777c65dbb7d2a0634bc1ea
6,19.90,00054e8431b9d7675808bcb819fb4a32
7,810.00,000576fe39319847cbb9d288c5617fa6
8,145.95,0005a1a1728c9d785b8e2b08b904576c
9,53.99,0005f50442cb953dcd1d21e1fb923495


In [20]:
pd.read_sql("""
SELECT
    fs.order_id,
    fs.price,
    fs.total_payment,
    ot.total_merchandise,
    (fs.price * 1.0 / ot.total_merchandise) AS ratio,
    ROUND(
        (fs.price * 1.0 / ot.total_merchandise) * fs.total_payment,
        2
    ) AS allocated_payment_calc
FROM fact_sales_final fs
LEFT JOIN order_totals ot
ON fs.order_id = ot.order_id
LIMIT 10;
""", conn)

,order_id,price,total_payment,total_merchandise,ratio,allocated_payment_calc
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,72.19,58.90,1.0,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,259.83,239.90,1.0,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,216.87,199.00,1.0,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,25.78,12.99,1.0,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,218.04,199.90,1.0,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,34.59,21.90,1.0,34.59
6,00054e8431b9d7675808bcb819fb4a32,19.90,31.75,19.90,1.0,31.75
7,000576fe39319847cbb9d288c5617fa6,810.00,880.75,810.00,1.0,880.75
8,0005a1a1728c9d785b8e2b08b904576c,145.95,157.60,145.95,1.0,157.60
9,0005f50442cb953dcd1d21e1fb923495,53.99,65.39,53.99,1.0,65.39


In [21]:
pd.read_sql("""
PRAGMA table_info(fact_sales_final_v3);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,order_item_id,INT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,seller_id,TEXT,0,None,0
4,4,customer_id,TEXT,0,None,0
5,5,customer_unique_id,TEXT,0,None,0
6,6,customer_city,TEXT,0,None,0
7,7,customer_state,TEXT,0,None,0
8,8,purchase_date,,0,None,0
9,9,order_purchase_timestamp,TEXT,0,None,0


In [22]:
conn.executescript("""

DROP TABLE IF EXISTS fact_sales;

CREATE TABLE fact_sales AS

SELECT

    fs.*,

    ROUND(
        (fs.price * 1.0 / ot.total_merchandise)
        * fs.total_payment,
        2
    ) AS allocated_payment_new

FROM fact_sales_final fs

LEFT JOIN order_totals ot
ON fs.order_id = ot.order_id;

""")

conn.commit()

In [23]:
pd.read_sql("""
SELECT
allocated_payment_new
FROM fact_sales
LIMIT 10;
""", conn)

,allocated_payment_new
0,72.19
1,259.83
2,216.87
3,25.78
4,218.04
5,34.59
6,31.75
7,880.75
8,157.60
9,65.39


In [24]:
fact = pd.read_sql("""
SELECT * FROM fact_sales
""", conn)

fact.to_csv("../data/tableau/fact_sales.csv", index=False)

print(fact.shape)

(112834, 30)
